In [1]:
from dotenv import load_dotenv
from anthropic import Anthropic


In [2]:
load_dotenv()
import os 
client = Anthropic()


## Get Current Datetime 




In [4]:
from anthropic.types import Message
messages = [] 
def add_user_message(messages, message):
    user_message = {
        "role" : "user",
        "content" : message.content if isinstance(message, Message) else message 
    }

    messages.append(user_message)


def add_assistant_message(messages, message):
    assistant_message = {
        "role" : "assistant",
        "content" : message.content if isinstance(message, Message) else message 
    }

    messages.append(assistant_message)

In [5]:
from  datetime import *
from anthropic.types import ToolParam

model = "claude-sonnet-4-6"


def get_current_datetime(date_time_format="%Y-%m-%d %H:%M:%S"):

    if not date_time_format:
        raise ValueError("Date Format can not be empty")
    return datetime.now().strftime(date_time_format)


# Write Schema for above funtion (this is the mertadata about what function is , its job etc. this is fed to Claude at runtime to make it more aware 
# about this function.)

get_current_datetime_schema = ToolParam({
        "name": "get_current_datetime",
        "description": "Get the current date and time, formatted according to a given strftime-style format string. Defaults to 'YYYY-MM-DD HH:MM:SS' format if no format is specified.",
        "input_schema": {
            "type": "object",
            "properties": {
                "date_time_format": {
                    "type": "string",
                    "description": "A Python strftime format string used to format the current date/time (e.g. '%Y-%m-%d %H:%M:%S', '%B %d, %Y', '%H:%M'). Defaults to '%Y-%m-%d %H:%M:%S'."
                }
            },
            "required": []
        }
    })



# messages.append({
#     "role" : "user",
#     "content" : "What is the time now formatted in HH:MM:SS ?"
#  })

# pass this to model using chat 

# response = client.messages.create(
#     model = model,
#     max_tokens = 100,
#     messages = messages,
#     tools=[get_current_datetime_schema]
# )



In [70]:
response

Message(id='msg_011CfNMT7ia5Vsdj5La8sH3R', container=None, content=[ToolUseBlock(id='toolu_01QoB3pKk3GduwoRqDkxEJLB', caller=DirectCaller(type='direct'), input={'date_time_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use', toolset_name=None)], model='claude-sonnet-4-6', role='assistant', stop_details=None, stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=680, output_tokens=65, output_tokens_details=None, server_tool_use=None, service_tier='standard'), diagnostics=None)

In [39]:
# add_user_message(messages, "What is time today ?")

# response = client.messages.create(
#     model = model,
#     max_tokens = 100,
#     messages = messages,
#     tools=[get_current_datetime_schema]
# )

# response

# add_assistant_message(messages, response)


# Lets modify above function to reusable chat function 

def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model" : model,
        "max_tokens" : 1000,
        "messages" : messages,
        # "extra_body" : {"temperature" : temperature},
        "stop_sequences" : stop_sequences,
    }

    # Add tool if given 
    if tools:
        params['tools'] = tools
    
    # Add systems prompt if given 
    if system: 
        params['system'] = system

    response = client.messages.create(**params)

    return response


def text_from_message(response):
    return "\n".join(
        [block.text for block in response.content if block.type == "text"]
    )

# simple join method demo 
# it alwasy works on strings , iterables 
ll = ['sachin', "10", 'jagtap']

dd = "--".join(each for each in ll) 
dd

'sachin--10--jagtap'

In [30]:
add_user_message(messages, "what is time now")
op = chat(messages=messages, tools=[get_current_datetime_schema])
op.content



[TextBlock(citations=None, text='Let me check the current time for you right now!', type='text'),
 ToolUseBlock(id='toolu_01ARGABKoP32WjpTgNTq7yKD', caller=DirectCaller(type='direct'), input={'date_time_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use', toolset_name=None)]

In [42]:
# create function that will keep calling calude , untill claude has final answer 
import json 


def run_tool(tool_name, tool_inputs):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_inputs)

def run_tools(message):
    tool_requests = [
        block for block in message.content if block.type == 'tool_use'
    ]
    

    tool_results_block = []

    for tool_request in tool_requests:
        
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)

            tool_result_block = {
                "type" : "tool_result",
                "tool_use_id" : tool_request.id,
                "content" : json.dumps(tool_output),
                "is_error" : False
            }
        except Exception as e:
            tool_result_block = {
                "type" : "tool_result",
                "tool_use_id" : tool_requests.id,
                "content" : "Error {e}",
                "is_error" : True
            }
        tool_results_block.append(tool_result_block)

    return tool_results_block
            



            
def run_conversation(messages):
    while True:
        response = chat(messages, tools=[get_current_datetime_schema])

        # Add claude response t othe list 

        add_assistant_message(messages, response)
        print(text_from_message(response))

        # check if it has stop reason 
        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)
    
    return messages 

In [43]:
messages = [] 
add_user_message(messages, "What is the current time in HH:MM ? Also time in SS ?")
run_conversation(messages)



I'll fetch both the current time in HH:MM format and the seconds simultaneously!
Here are the current time details:

- 🕐 **Time (HH:MM):** 22:04
- ⏱️ **Seconds (SS):** 24

So the full current time is **22:04:24**!


[{'role': 'user',
  'content': 'What is the current time in HH:MM ? Also time in SS ?'},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll fetch both the current time in HH:MM format and the seconds simultaneously!", type='text'),
   ToolUseBlock(id='toolu_01RMfm83L5Vxom1EDvVAUeQj', caller=DirectCaller(type='direct'), input={'date_time_format': '%H:%M'}, name='get_current_datetime', type='tool_use', toolset_name=None),
   ToolUseBlock(id='toolu_012GPoa2ZLMnT9cX6yhau8FC', caller=DirectCaller(type='direct'), input={'date_time_format': '%S'}, name='get_current_datetime', type='tool_use', toolset_name=None)]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_01RMfm83L5Vxom1EDvVAUeQj',
    'content': '"22:04"',
    'is_error': False},
   {'type': 'tool_result',
    'tool_use_id': 'toolu_012GPoa2ZLMnT9cX6yhau8FC',
    'content': '"24"',
    'is_error': False}]},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text='Here 

In [41]:
messages

[{'role': 'user',
  'content': 'What is the current time in HH:MM ? Also time in SS ?'},
 {'role': 'assistant',
  'content': [TextBlock(citations=None, text="I'll fetch both the current time in HH:MM format and the seconds simultaneously!", type='text'),
   ToolUseBlock(id='toolu_01Ybj26QuKMJZEUgTXqpGRyd', caller=DirectCaller(type='direct'), input={'date_time_format': '%H:%M'}, name='get_current_datetime', type='tool_use', toolset_name=None),
   ToolUseBlock(id='toolu_01FmLGJBfV6w3TnNXg7JRabr', caller=DirectCaller(type='direct'), input={'date_time_format': '%S'}, name='get_current_datetime', type='tool_use', toolset_name=None)]},
 {'role': 'user', 'content': None}]

'sachin--10--jagtap'

In [ ]:
result

In [ ]:
Message(id='msg_011CfKU3QkPPNDgauXwUQeME', container=None, content=[ToolUseBlock(id='toolu_01GpdPWig59qUaGbPNM6Gkzv', caller=DirectCaller(type='direct'), input={'date_time_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use', toolset_name=None)], model='claude-sonnet-4-6', role='assistant', stop_details=None, stop_reason='tool_use', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=680, output_tokens=65, output_tokens_details=None, server_tool_use=None, service_tier='standard'))

In [ ]:
[ToolUseBlock(id='toolu_01JrQewCxRERwLaCEgBL8TZm', caller=DirectCaller(type='direct'), input={'date_time_format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use', toolset_name=None)]

In [37]:
datetime.now()

datetime.datetime(2026, 9, 23, 1, 35, 28, 316390)

In [15]:
print(ANTHROPIC_API_KEY)

NameError: name 'ANTHROPIC_API_KEY' is not defined